In [9]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

# === Load JSON file ===
json_file = "manual_scores_Final.json"  # Replace with your JSON filename
with open(json_file, "r") as f:
    data = json.load(f)

In [10]:
# === Threshold to consider a question "good" ===
threshold = 4.0

# === Prepare lists to store results ===
file_metrics = []

In [12]:
# === Compute per-file metrics and F1/Precision/Recall ===
for file_name, questions in data.items():
    metrics_list = []
    y_true = []
    y_pred = []
    
    for q in questions:
        scores = q["scores"]
        overall = sum(scores.values()) / len(scores)
        metrics_list.append({**scores, "Overall": overall})
        
        # Manual label: assume scores >= threshold are "good"
        y_true.append(1 if overall >= threshold else 0)
        
        # Predicted label: model generated all questions, assume all predicted "good" (1)
        # Modify if you have model confidence for prediction
        y_pred.append(1)
    
    df = pd.DataFrame(metrics_list)
    avg_metrics = df.mean().to_dict()
    
    # Compute F1 metrics
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    avg_metrics.update({
        "File": file_name,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })
    
    file_metrics.append(avg_metrics)

# === Convert to DataFrame ===
file_metrics_df = pd.DataFrame(file_metrics)
cols = ["File", "Relevance", "Specificity", "Depth", "clarity", "Overall", "Precision", "Recall", "F1"]
file_metrics_df = file_metrics_df[cols]

print("=== Per-File Metrics with F1 ===")
print(file_metrics_df)

=== Per-File Metrics with F1 ===
               File  Relevance  Specificity     Depth   clarity   Overall  \
0        .gitignore   4.750000     3.500000  2.250000  5.000000  3.875000   
1     calculator.py   3.750000     3.250000  2.750000  4.250000  3.500000   
2   configLoader.js   4.142857     3.285714  2.428571  4.571429  3.607143   
3             db.js   5.000000     3.600000  3.000000  5.000000  4.150000   
4          index.js   4.500000     4.400000  3.900000  4.800000  4.400000   
5      package.json   4.428571     4.571429  3.428571  5.000000  4.357143   
6        .gitignore   4.750000     3.500000  2.250000  5.000000  3.875000   
7     calculator.py   3.750000     3.250000  2.750000  4.250000  3.500000   
8   configLoader.js   4.142857     3.285714  2.428571  4.571429  3.607143   
9             db.js   5.000000     3.600000  3.000000  5.000000  4.150000   
10         index.js   4.500000     4.400000  3.900000  4.800000  4.400000   
11     package.json   4.428571     4.571429

In [13]:

# === Compute global metrics ===
all_scores = []
y_true_global = []
y_pred_global = []

for questions in data.values():
    for q in questions:
        scores = q["scores"]
        overall = sum(scores.values()) / len(scores)
        all_scores.append({**scores, "Overall": overall})
        
        y_true_global.append(1 if overall >= threshold else 0)
        y_pred_global.append(1)

global_df = pd.DataFrame(all_scores)
global_avg_scores = global_df.mean().to_dict()

precision_global = precision_score(y_true_global, y_pred_global)
recall_global = recall_score(y_true_global, y_pred_global)
f1_global = f1_score(y_true_global, y_pred_global)

print("\n=== Global Metrics with F1 ===")
for metric, value in global_avg_scores.items():
    print(f"{metric}: {value:.2f}")
print(f"Precision: {precision_global:.2f}")
print(f"Recall: {recall_global:.2f}")
print(f"F1 Score: {f1_global:.2f}")


=== Global Metrics with F1 ===
Relevance: 4.37
Specificity: 3.83
Depth: 3.07
clarity: 4.73
Overall: 4.00
Precision: 0.61
Recall: 1.00
F1 Score: 0.76


In [ ]:
# === Optional Visualization ===
plt.figure(figsize=(10,6))
plt.bar(file_metrics_df["File"], file_metrics_df["F1"], color="lightgreen")
plt.title("F1 Score per File")
plt.ylabel("F1 Score")
plt.xticks(rotation=45)
plt.ylim(0,1)
plt.tight_layout()
plt.show()